# NorDex Manufacturing Data Cleaning and Preparation

## Week 1 — Day 5

This notebook transforms the raw manufacturing extract into Clean Dataset Version 1. The process includes data-type conversion, missing-value treatment, category standardisation, duplicate removal, anomaly flagging and final validation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
project_folder = Path.cwd()

print("Notebook location:", project_folder)
print("CSV files found:")

for file in project_folder.glob("*.csv"):
    print("-", file.name)

Notebook location: c:\Users\User\AMDARI Project 1
CSV files found:
- extracted dataset file.csv
- raw_manufacturing_dataset.csv


In [3]:
data_file = project_folder / "raw_manufacturing_dataset.csv"

if not data_file.exists():
    raise FileNotFoundError(
        f"Could not find {data_file.name} inside {project_folder}"
    )

manufacturing_data = pd.read_csv(data_file)

print("File loaded successfully.")
print("Dataset shape:", manufacturing_data.shape)

manufacturing_data.head()

File loaded successfully.
Dataset shape: (10117, 31)


,shift_id,shift_name,start_time,end_time,supervisor_id,production_id,date,units_produced,defect_count,cycle_time_avg,shift_efficiency_score,operator_id,operator_name,experience_level,skill_category,machine_id,runtime_hours,downtime_minutes,maintenance_flag,machine_status,maintenance_id,issue_type,maintenance_downtime,resolved_by,qc_id,defect_type,severity,inspection_result,temperature,humidity,timestamp
0,1,Morning,06:00:00.0000000,14:00:00.0000000,SUP_01,1,2024-01-01,929,20,35.65,95.945255,OP_019,Operator_19,7,Expert,MC_001,7.24,15.39,0,Operational,NaN,NaN,NaN,NaN,1,Material,Minor,Rework,22.3,49.9,2024-01-01 08:00:00
1,1,Morning,06:00:00.0000000,14:00:00.0000000,SUP_01,1,2024-01-01,929,20,35.65,95.945255,OP_019,Operator_19,7,Expert,MC_001,7.24,15.39,0,Operational,NaN,NaN,NaN,NaN,2,Assembly,Minor,Accepted,22.3,49.9,2024-01-01 08:00:00
2,1,Morning,06:00:00.0000000,14:00:00.0000000,SUP_01,1,2024-01-01,929,20,35.65,95.945255,OP_019,Operator_19,7,Expert,MC_001,7.24,15.39,0,Operational,NaN,NaN,NaN,NaN,3,Dimensional,Minor,Rework,22.3,49.9,2024-01-01 08:00:00
3,1,Morning,06:00:00.0000000,14:00:00.0000000,SUP_01,1,2024-01-01,929,20,35.65,95.945255,OP_019,Operator_19,7,Expert,MC_001,7.24,15.39,0,Operational,NaN,NaN,NaN,NaN,4,Dimensional,Minor,Rework,22.3,49.9,2024-01-01 08:00:00
4,1,Morning,06:00:00.0000000,14:00:00.0000000,SUP_01,1,2024-01-01,929,20,35.65,95.945255,OP_019,Operator_19,7,Expert,MC_001,7.24,15.39,0,Operational,NaN,NaN,NaN,NaN,5,Cosmetic,Minor,Rework,22.3,49.9,2024-01-01 08:00:00


In [4]:
raw_data = manufacturing_data.copy(deep=True)
clean_data = manufacturing_data.copy(deep=True)

print("Raw dataset shape:", raw_data.shape)
print("Working copy shape:", clean_data.shape)

Raw dataset shape: (10117, 31)
Working copy shape: (10117, 31)


In [5]:
before_cleaning = pd.DataFrame({
    "Measure": [
        "Rows",
        "Columns",
        "Exact duplicate rows",
        "Total missing values"
    ],
    "Before cleaning": [
        clean_data.shape[0],
        clean_data.shape[1],
        clean_data.duplicated().sum(),
        clean_data.isna().sum().sum()
    ]
})

before_cleaning

,Measure,Before cleaning
0,Rows,10117
1,Columns,31
2,Exact duplicate rows,0
3,Total missing values,28507


In [6]:
clean_data.columns = (
    clean_data.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print("Column names standardised.")
print(clean_data.columns.tolist())

Column names standardised.
['shift_id', 'shift_name', 'start_time', 'end_time', 'supervisor_id', 'production_id', 'date', 'units_produced', 'defect_count', 'cycle_time_avg', 'shift_efficiency_score', 'operator_id', 'operator_name', 'experience_level', 'skill_category', 'machine_id', 'runtime_hours', 'downtime_minutes', 'maintenance_flag', 'machine_status', 'maintenance_id', 'issue_type', 'maintenance_downtime', 'resolved_by', 'qc_id', 'defect_type', 'severity', 'inspection_result', 'temperature', 'humidity', 'timestamp']


In [7]:
categorical_columns = clean_data.select_dtypes(
    include="object"
).columns

excluded_time_columns = [
    "date",
    "start_time",
    "end_time",
    "timestamp"
]

categorical_columns = [
    column for column in categorical_columns
    if column not in excluded_time_columns
]

for column in categorical_columns:
    clean_data[column] = (
        clean_data[column]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Text spacing standardised.")

C:\Users\User\AppData\Local\Temp\ipykernel_14716\2058523160.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = clean_data.select_dtypes(


Text spacing standardised.


In [8]:
for column in [
    "shift_name",
    "skill_category",
    "machine_status",
    "issue_type",
    "defect_type",
    "severity",
    "inspection_result"
]:
    print(f"\n{column}:")
    print(sorted(clean_data[column].dropna().unique()))


shift_name:
['Evening', 'Morning', 'Night']

skill_category:
['Expert', 'Intermediate', 'Junior', 'Senior']

machine_status:
['Issues', 'Operational']

issue_type:
['Calibration', 'Electrical', 'Mechanical', 'Preventive', 'Software']

defect_type:
['Assembly', 'Cosmetic', 'Dimensional', 'Material', 'Surface']

severity:
['Critical', 'Major', 'Minor']

inspection_result:
['Accepted', 'Rejected', 'Rework']


In [9]:
clean_data["date"] = pd.to_datetime(
    clean_data["date"],
    errors="coerce"
)

clean_data["start_datetime"] = pd.to_datetime(
    clean_data["date"].dt.strftime("%Y-%m-%d")
    + " "
    + clean_data["start_time"].astype(str),
    errors="coerce"
)

clean_data["end_datetime"] = pd.to_datetime(
    clean_data["date"].dt.strftime("%Y-%m-%d")
    + " "
    + clean_data["end_time"].astype(str),
    errors="coerce"
)

In [10]:
overnight_shifts = (
    clean_data["end_datetime"]
    <= clean_data["start_datetime"]
)

clean_data.loc[
    overnight_shifts,
    "end_datetime"
] += pd.Timedelta(days=1)

print("Overnight shifts corrected:", overnight_shifts.sum())

Overnight shifts corrected: 3552


In [11]:
clean_data["timestamp"] = pd.to_datetime(
    clean_data["timestamp"],
    errors="coerce"
)

print(clean_data[
    [
        "date",
        "start_datetime",
        "end_datetime",
        "timestamp"
    ]
].dtypes)

date              datetime64[us]
start_datetime    datetime64[ns]
end_datetime      datetime64[ns]
timestamp         datetime64[us]
dtype: object


In [12]:
rows_before_duplicates = len(clean_data)
duplicate_count = clean_data.duplicated().sum()

clean_data = (
    clean_data
    .drop_duplicates()
    .reset_index(drop=True)
)

rows_after_duplicates = len(clean_data)

print("Rows before:", rows_before_duplicates)
print("Exact duplicates found:", duplicate_count)
print("Rows after:", rows_after_duplicates)

Rows before: 10117
Exact duplicates found: 0
Rows after: 10117


In [13]:
pd.crosstab(
    clean_data["maintenance_flag"],
    clean_data["maintenance_id"].isna(),
    margins=True
)

maintenance_id,False,True,All
maintenance_flag,,,
0,2058,6223,8281
1,1836,0,1836
All,3894,6223,10117


In [14]:
no_maintenance = clean_data["maintenance_flag"] == 0

In [15]:
clean_data.loc[
    no_maintenance & clean_data["issue_type"].isna(),
    "issue_type"
] = "No maintenance"

clean_data.loc[
    no_maintenance & clean_data["resolved_by"].isna(),
    "resolved_by"
] = "Not applicable"

clean_data.loc[
    no_maintenance & clean_data["maintenance_downtime"].isna(),
    "maintenance_downtime"
] = 0

In [16]:
clean_data["maintenance_id"] = (
    clean_data["maintenance_id"]
    .astype("Int64")
)

In [ ]:
clean_data["temperature_was_missing"] = (
    clean_data["temperature"].isna().astype("int8")
)

clean_data["humidity_was_missing"] = (
    clean_data["humidity"].isna().astype("int8")
)

In [18]:
temperature_median = clean_data["temperature"].median()
humidity_median = clean_data["humidity"].median()

clean_data["temperature"] = (
    clean_data["temperature"]
    .fillna(temperature_median)
)

clean_data["humidity"] = (
    clean_data["humidity"]
    .fillna(humidity_median)
)

print("Temperature median used:", temperature_median)
print("Humidity median used:", humidity_median)

Temperature median used: 19.7
Humidity median used: 45.1


In [19]:
clean_data["timestamp_was_missing"] = (
    clean_data["timestamp"].isna().astype("int8")
)

In [20]:
clean_data["timestamp"] = (
    clean_data["timestamp"]
    .fillna(clean_data["start_datetime"])
)

print(
    "Remaining missing timestamps:",
    clean_data["timestamp"].isna().sum()
)

Remaining missing timestamps: 0


In [21]:
clean_data["efficiency_above_100_flag"] = (
    clean_data["shift_efficiency_score"] > 100
).astype("int8")

clean_data["invalid_production_flag"] = (
    (clean_data["units_produced"] < 0)
    | (clean_data["defect_count"] < 0)
    | (
        clean_data["defect_count"]
        > clean_data["units_produced"]
    )
).astype("int8")

clean_data["invalid_runtime_flag"] = (
    (clean_data["runtime_hours"] < 0)
    | (clean_data["runtime_hours"] > 8)
).astype("int8")

clean_data["invalid_downtime_flag"] = (
    (clean_data["downtime_minutes"] < 0)
    | (clean_data["downtime_minutes"] > 480)
).astype("int8")

clean_data["invalid_humidity_flag"] = (
    (clean_data["humidity"] < 0)
    | (clean_data["humidity"] > 100)
).astype("int8")

In [22]:
anomaly_summary = pd.Series({
    "Efficiency above 100":
        clean_data["efficiency_above_100_flag"].sum(),

    "Invalid production records":
        clean_data["invalid_production_flag"].sum(),

    "Invalid runtime records":
        clean_data["invalid_runtime_flag"].sum(),

    "Invalid downtime records":
        clean_data["invalid_downtime_flag"].sum(),

    "Invalid humidity records":
        clean_data["invalid_humidity_flag"].sum()
})

anomaly_summary

Efficiency above 100          134
Invalid production records      0
Invalid runtime records         0
Invalid downtime records        0
Invalid humidity records        0
dtype: int64

In [23]:
remaining_missing = pd.DataFrame({
    "missing_count": clean_data.isna().sum(),
    "missing_percentage": (
        clean_data.isna().mean() * 100
    ).round(2)
}).query("missing_count > 0")

remaining_missing

,missing_count,missing_percentage
maintenance_id,6223,61.51


In [24]:
cleaning_summary = pd.DataFrame({
    "Measure": [
        "Rows",
        "Columns",
        "Exact duplicate rows",
        "Total missing values",
        "Temperature missing",
        "Humidity missing",
        "Timestamp missing",
        "Efficiency above 100"
    ],

    "Before cleaning": [
        len(raw_data),
        raw_data.shape[1],
        raw_data.duplicated().sum(),
        raw_data.isna().sum().sum(),
        raw_data["temperature"].isna().sum(),
        raw_data["humidity"].isna().sum(),
        raw_data["timestamp"].isna().sum(),
        (
            raw_data["shift_efficiency_score"] > 100
        ).sum()
    ],

    "After cleaning": [
        len(clean_data),
        clean_data.shape[1],
        clean_data.duplicated().sum(),
        clean_data.isna().sum().sum(),
        clean_data["temperature"].isna().sum(),
        clean_data["humidity"].isna().sum(),
        clean_data["timestamp"].isna().sum(),
        clean_data["efficiency_above_100_flag"].sum()
    ]
})

cleaning_summary

,Measure,Before cleaning,After cleaning
0,Rows,10117,10117
1,Columns,31,41
2,Exact duplicate rows,0,0
3,Total missing values,28507,6223
4,Temperature missing,1205,0
5,Humidity missing,1205,0
6,Timestamp missing,1205,0
7,Efficiency above 100,134,134


In [ ]:
output_file = (
    project_folder
    / "cleaned_manufacturing_dataset_v1.csv"
)

clean_data.to_csv(
    output_file,
    index=False
)

print("Clean dataset exported successfully.")
print("Output file:", output_file)
print("File exists:", output_file.exists())
print(
    "File size:",
    round(output_file.stat().st_size / 1024**2, 2),
    "MB"
)

Clean dataset exported successfully.
Output file: c:\Users\User\AMDARI Project 1\cleaned_manufacturing_dataset_v1.csv
File exists: True
File size: 2.94 MB


: 